# CCA-F 考试复习 Notebook — Claude Code

本 notebook 按你给定的 CCA-F task 列表重写。每个 Task 都包含概念解释、可运行 Python 示例、考点总结和 2 道模拟题。

## 环境初始化

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

MODEL = "claude-haiku-4-5-20251001"
print("环境就绪：本 notebook 使用本地模拟，代码结构贴近 Anthropic Messages API / Claude Code 工作流。")

## D3.1 — 配置 CLAUDE.md 层级与 YAML frontmatter 路径作用域

### 📖 概念解释

`CLAUDE.md` 是 Claude Code 读取长期项目指令的主要入口。CCA-F 常考三件事：层级、路径作用域、以及 YAML frontmatter。一般应区分个人级规则、仓库级规则和目录级规则；越靠近目标文件的规则越具体，但不应把临时任务或个人偏好写进团队仓库。路径作用域用于让不同子树使用不同约束，例如 `src/api/**` 强制输入校验，`docs/**` 强制文档风格。

YAML frontmatter 的价值不是“让 Markdown 更好看”，而是把规则元数据结构化：名称、适用路径、排除路径、优先级或说明可以被工具稳定读取。考试陷阱通常是把全局 `~/.claude/CLAUDE.md` 当成团队共享配置，或认为目录级规则会自动适用于所有仓库文件。

In [ ]:
# Task 3.1：CLAUDE.md 层级、路径作用域与 YAML frontmatter
# 本示例只模拟配置解析，不需要安装 Claude Code。

import fnmatch
from dataclasses import dataclass

claude_md_files = {
    "~/.claude/CLAUDE.md": """---
name: personal-defaults
applies_to: ['**/*']
---
- 使用我个人偏好的简短回答风格。
""",
    "./CLAUDE.md": """---
name: project-standards
applies_to: ['**/*']
---
- 遵守团队架构边界，修改前先阅读相邻测试。
""",
    "./src/api/CLAUDE.md": """---
name: api-standards
applies_to: ['src/api/**/*.py', 'src/api/*.py']
exclude: ['src/api/experimental/**']
---
- API handler 必须验证输入、返回结构化错误并覆盖失败路径测试。
""",
}

def parse_frontmatter(markdown: str) -> tuple[dict, str]:
    """解析本 notebook 示例使用的简单 YAML frontmatter。"""
    if not markdown.startswith("---\n"):
        return {}, markdown
    _, raw_meta, body = markdown.split("---\n", 2)
    meta = {}
    for line in raw_meta.splitlines():
        key, value = line.split(":", 1)
        value = value.strip()
        if value.startswith("["):
            meta[key.strip()] = [item.strip().strip("'") for item in value.strip("[]").split(",") if item.strip()]
        else:
            meta[key.strip()] = value
    return meta, body.strip()

@dataclass(frozen=True)
class ScopedRule:
    source: str
    level: int
    name: str
    applies_to: tuple[str, ...]
    exclude: tuple[str, ...]
    body: str

def load_rules(files: dict[str, str]) -> list[ScopedRule]:
    levels = {"~/.claude/CLAUDE.md": 0, "./CLAUDE.md": 1, "./src/api/CLAUDE.md": 2}
    loaded = []
    for source, markdown in files.items():
        meta, body = parse_frontmatter(markdown)
        loaded.append(ScopedRule(
            source=source,
            level=levels[source],
            name=meta["name"],
            applies_to=tuple(meta.get("applies_to", ["**/*"])),
            exclude=tuple(meta.get("exclude", [])),
            body=body,
        ))
    return sorted(loaded, key=lambda rule: rule.level)

def active_rule_names(path: str) -> list[str]:
    active = []
    for rule in load_rules(claude_md_files):
        included = any(fnmatch.fnmatch(path, pattern) for pattern in rule.applies_to)
        excluded = any(fnmatch.fnmatch(path, pattern) for pattern in rule.exclude)
        if included and not excluded:
            active.append(rule.name)
    return active

for candidate in ["src/api/refund.py", "src/api/experimental/prototype.py", "docs/guide.md"]:
    print(candidate, "=>", active_rule_names(candidate))

### ⚠️ 考点总结 & 易错点

**正确做法**

- 把团队规则放进仓库级或目录级 `CLAUDE.md`，把个人偏好留在用户级配置。
- 用 frontmatter 或等价结构声明 `applies_to` / `exclude`，让路径作用域可检查、可维护。
- 将长期、稳定、跨任务有效的约定写入配置；临时任务要求留在当前 prompt 或 issue 中。

**常见陷阱**

- 误以为全局 `~/.claude/CLAUDE.md` 会随项目共享给团队。
- 只写自然语言“请遵守 API 规范”，却没有限定哪些路径适用。
- 把目录级规则理解成覆盖所有上层规则；更准确的判断是规则叠加，局部规则更具体。

### 🧪 模拟题

**Q1.** 一个仓库中 `src/api/` 和 `docs/` 需要不同 Claude Code 指令。最合适的配置方式是什么？

A) 在项目/目录 `CLAUDE.md` 中用路径作用域或 frontmatter 限定规则  
B) 把所有规则写入个人 `~/.claude/CLAUDE.md`  
C) 每次运行前手动粘贴两个目录的全部规则  
D) 只依赖模型根据文件名自行推断

> **答案：A**  
> **解析：** CCA-F 强调可维护、可共享的项目配置。路径作用域能让不同目录自动获得对应规则。

**Q2.** 哪一项最适合作为仓库级 `CLAUDE.md` 的内容？

A) “今天帮我改一下支付页面文案”  
B) “在 `src/api/**` 修改 handler 时必须保留输入校验和失败路径测试”  
C) “我个人喜欢所有回答都不超过三句话”  
D) “忽略本仓库的 lint 和测试失败”

> **答案：B**  
> **解析：** 仓库级规则应是长期、团队共享、可验证的工程约定；临时任务和个人偏好不应污染项目配置。

## D3.2 — 创建自定义 Slash Commands 并管理 .claude/commands/

### 📖 概念解释

Slash command 适合把高频、可重复的团队 prompt 固化为命令，通常放在仓库的 `.claude/commands/` 下，例如 `.claude/commands/review-api.md`。它的重点是快速触发一致的工作流：参数说明、输出格式、检查清单和完成标准。团队共享命令应进入版本控制；个人命令才放在用户目录。

`SKILL.md` 适合更完整的能力单元，常带 YAML frontmatter，例如 `name`、`description`、`argument-hint`、`allowed-tools` 和上下文隔离策略。CCA-F 重点不是背文件名，而是理解边界：slash command 是调用入口，Skill 是封装能力；Skill 应最小授权，避免把无关上下文和过宽工具暴露给任务。

In [ ]:
# Task 3.2：slash command、.claude/commands 与 Skill 隔离
# 本示例模拟命令注册和 Skill frontmatter 校验，不调用 Claude Code。

import re

project_files = {
    ".claude/commands/review-api.md": """# /review-api
Argument: $ARGUMENTS

Review the API file for auth, validation, structured errors, and tests.
Return findings as severity, file, line, and fix.
""",
    "skills/api-review/SKILL.md": """---
name: api-review
description: Review API handlers with isolated context.
argument-hint: path to an API module
allowed-tools: [Read, Grep]
context: fork
---

Inspect only the requested API module and adjacent tests.
""",
}

def discover_slash_commands(files: dict[str, str]) -> dict[str, str]:
    commands = {}
    for path, body in files.items():
        if path.startswith(".claude/commands/") and path.endswith(".md"):
            command_name = "/" + path.rsplit("/", 1)[-1].removesuffix(".md")
            commands[command_name] = body
    return commands

def parse_inline_frontmatter(markdown: str) -> dict[str, object]:
    meta_text = markdown.split("---", 2)[1]
    meta = {}
    for line in meta_text.splitlines():
        if not line.strip():
            continue
        key, value = line.split(":", 1)
        value = value.strip()
        if value.startswith("["):
            meta[key] = [item.strip() for item in value.strip("[]").split(",")]
        else:
            meta[key] = value
    return meta

def skill_is_exam_ready(skill_markdown: str) -> bool:
    meta = parse_inline_frontmatter(skill_markdown)
    required = {"name", "description", "argument-hint", "allowed-tools", "context"}
    least_privilege = set(meta.get("allowed-tools", [])) <= {"Read", "Grep", "Glob"}
    isolated = meta.get("context") in {"fork", "isolated"}
    return required <= set(meta) and least_privilege and isolated

commands = discover_slash_commands(project_files)
print("可用命令:", sorted(commands))
print("/review-api 包含参数占位符:", "$ARGUMENTS" in commands["/review-api"])
print("Skill 最小授权且隔离:", skill_is_exam_ready(project_files["skills/api-review/SKILL.md"]))

### ⚠️ 考点总结 & 易错点

**正确做法**

- 团队共享 slash command 放在仓库 `.claude/commands/`，并写清参数、输出格式和完成标准。
- 复杂复用能力放入 `SKILL.md`，用 frontmatter 声明能力、参数提示、允许工具和上下文隔离。
- Skill 授权遵循最小权限；能用 `Read/Grep/Glob` 完成的任务，不应默认开放 shell 或网络。

**常见陷阱**

- 把团队命令放到个人 `~/.claude/commands/`，导致 CI 或队友不可复现。
- 把 slash command 与 Skill 混为一谈：前者是快捷入口，后者是能力封装。
- Skill 没有参数提示、输出规范或隔离策略，造成上下文污染和过宽权限。

### 🧪 模拟题

**Q1.** 团队希望所有成员都能使用同一个 `/review-api` 命令。该文件应放在哪里？

A) 仓库的 `.claude/commands/review-api.md`  
B) 每个成员的 `~/.claude/commands/review-api.md`  
C) 根目录 `README.md` 的某一节  
D) CI 平台的 secrets 配置页

> **答案：A**  
> **解析：** 仓库级 `.claude/commands/` 可版本控制、可共享、可审查。

**Q2.** 哪个 `SKILL.md` frontmatter 设计最符合考试要求？

A) 只写 `name`，正文里再说明所有细节  
B) 声明 `description`、`argument-hint`、最小 `allowed-tools`，并启用隔离上下文  
C) 默认开放 Bash、网络和所有仓库上下文，避免能力不足  
D) 不写 frontmatter，让模型自动决定工具和边界

> **答案：B**  
> **解析：** Skill 的重点是可发现、可约束、可隔离。过宽工具和隐式边界都是常见干扰项。

## D3.3 — 使用 Plan Mode 与会话管理命令

### 📖 概念解释

Plan Mode 用于先探索、提出方案并等待确认，适合多文件、高风险、架构边界不清的任务。它不是所有任务的强制步骤；单文件、低风险、目标明确的修复可以直接执行。CCA-F 常用场景判断题考查你是否能根据风险、影响范围和可恢复性选择工作流。

会话管理包含几个不同概念：`/memory` 保存长期有效的偏好或团队约定，不应用来存临时上下文；`/compact` 在长会话中压缩历史以保留关键决策；session continuity 关注跨会话恢复任务状态；`--print` 用于脚本和 CI 的非交互执行。不要把这些命令互相替代。

In [ ]:
# Task 3.3：Plan Mode、/memory、/compact、连续性与 --print
# 本示例根据任务风险生成建议工作流，不调用 Claude Code。

from dataclasses import dataclass, field

@dataclass
class ClaudeCodeSession:
    memory: list[str] = field(default_factory=list)
    transcript_tokens: int = 0
    decisions: list[str] = field(default_factory=list)

    def remember(self, fact: str, durable: bool) -> None:
        # /memory 只保存长期有效信息；临时 issue 细节不写入。
        if durable:
            self.memory.append(fact)

    def maybe_compact(self) -> str:
        if self.transcript_tokens > 150_000:
            kept = "; ".join(self.decisions[-3:]) or "保留当前目标和下一步"
            self.transcript_tokens = 40_000
            return f"/compact -> {kept}"
        return "无需 /compact"

def choose_execution_mode(files_touched: int, risk: str, in_ci: bool) -> str:
    if in_ci:
        return 'claude --print "run review and return JSON" --output-format json'
    if files_touched > 10 or risk == "high":
        return "Plan Mode: 先列方案、风险和验证步骤，等待确认"
    return "Direct execution: 范围清晰，直接修改并验证"

session = ClaudeCodeSession(transcript_tokens=180_000, decisions=["保留公共 API", "先补测试再改实现"])
session.remember("本仓库 API handler 必须覆盖失败路径测试", durable=True)
session.remember("当前 PR 编号是 1287", durable=False)

print("交互式大改:", choose_execution_mode(files_touched=18, risk="high", in_ci=False))
print("CI 调用:", choose_execution_mode(files_touched=3, risk="medium", in_ci=True))
print("压缩结果:", session.maybe_compact())
print("memory:", session.memory)

### ⚠️ 考点总结 & 易错点

**正确做法**

- 多文件、高风险、架构性变更先用 Plan Mode 明确方案和验证路径。
- `/compact` 处理长上下文压缩；`/memory` 只保存长期有效约定；session continuity 用于恢复任务状态。
- CI 或脚本使用 `--print` / `-p` 非交互执行，而不是等待用户输入。

**常见陷阱**

- 小修也强制进入冗长计划，降低效率。
- 把临时 issue 细节写入 `/memory`，造成后续任务污染。
- 用 `/compact` 当作计划工具，或用 Plan Mode 解决 CI 非交互问题。
- 会话过长时直接丢弃上下文，而不是压缩关键决策。

### 🧪 模拟题

**Q1.** 一个任务需要重构认证边界、影响二十多个文件，并可能改变公共 API。最合适的第一步是什么？

A) 进入 Plan Mode，先探索依赖、风险和验证方案  
B) 直接批量修改所有文件，再看测试结果  
C) 先 `/compact`，然后跳过方案确认  
D) 只运行格式化工具

> **答案：A**  
> **解析：** 高风险、多文件、架构性任务需要先规划和确认，避免不可控修改。

**Q2.** 哪项最适合写入 `/memory`？

A) “当前 issue 的临时复现步骤”  
B) “本仓库所有 API handler 必须包含失败路径测试”  
C) “刚才终端输出的完整日志”  
D) “下一条消息要回答什么”

> **答案：B**  
> **解析：** `/memory` 保存长期有效、跨会话有价值的团队约定；临时上下文应留在当前会话或任务记录中。

## D3.4 — 将 Claude Code 集成到 CI/CD 流水线

### 📖 概念解释

Claude Code 进入 CI/CD 后必须 headless、non-interactive、可解析、可失败。`--print` 或 `-p` 让命令执行后输出并退出；结构化输出（如 JSON）让流水线能读取结论、失败原因和修复建议。仅生成自然语言建议不等于 CI 集成，因为流水线无法稳定判断是否通过。

可靠的 CI/CD 设计还需要触发条件和验证闭环：例如 pull request 触发审查，测试失败时把失败摘要反馈给下一轮修复，最后检查退出码和测试结果。考试常见干扰项包括提高 temperature、增加 token、或把交互式 Claude Code 会话直接塞进 CI。

In [ ]:
# Task 3.4：Claude Code CI/CD、headless、--print 与验证循环
# 本示例模拟 CI 命令和测试反馈循环，不需要安装 Claude Code。

import json
import shlex

ci_workflow = {
    "on": ["pull_request"],
    "steps": [
        {
            "name": "claude-review",
            "run": 'claude --print "Review changed files and return JSON" --output-format json',
        },
        {
            "name": "tests",
            "run": "pytest -q",
        },
    ],
}

def is_non_interactive_claude_step(command: str) -> bool:
    tokens = shlex.split(command)
    return "claude" in tokens and ("--print" in tokens or "-p" in tokens) and "--output-format" in tokens and "json" in tokens

def simulate_test_verification_loop(test_results: list[bool]) -> dict[str, object]:
    attempts = []
    for attempt, passed in enumerate(test_results, start=1):
        attempts.append({"attempt": attempt, "tests_passed": passed})
        if passed:
            return {"status": "pass", "attempts": attempts, "exit_code": 0}
    return {"status": "fail", "attempts": attempts, "exit_code": 1}

review_step = ci_workflow["steps"][0]["run"]
result = simulate_test_verification_loop([False, False, True])

print("Claude step is CI-safe:", is_non_interactive_claude_step(review_step))
print(json.dumps(result, indent=2))
assert is_non_interactive_claude_step(review_step)
assert result["exit_code"] == 0

### ⚠️ 考点总结 & 易错点

**正确做法**

- CI 中使用 `claude --print` 或 `claude -p`，确保命令输出后退出。
- 要求 JSON 等结构化输出，并让流水线检查退出码、字段和测试结果。
- 用 pull request、push 或 scheduled job 等明确触发条件启动任务。
- 将测试失败摘要反馈到修复循环，并以验证结果决定通过或失败。

**常见陷阱**

- 在 CI 中启动交互式会话，导致作业挂起。
- 只输出自然语言审查意见，流水线无法稳定解析。
- 忽略退出码或测试结果，把“模型说已修复”当作通过条件。
- 试图用更高 temperature、更多 token 替代 headless 和验证设计。

### 🧪 模拟题

**Q1.** Claude Code 的 CI 作业一直挂起等待输入。最直接的修复是什么？

A) 使用 `claude --print` 或 `claude -p` 进行非交互执行  
B) 提高 temperature 让模型更主动  
C) 增加 max tokens 让模型有更多空间  
D) 把 stdin 指向开发者键盘

> **答案：A**  
> **解析：** CI/CD 必须 headless。`--print` / `-p` 是非交互脚本化执行的关键。

**Q2.** 哪个 CI 设计最符合 CCA-F 要求？

A) PR 触发 Claude Code 审查，JSON 输出，运行测试，失败摘要反馈给修复循环，最后检查退出码  
B) 手动打开 Claude Code，复制粘贴日志，再口头通知团队  
C) 只让模型生成“看起来没问题”的自然语言总结  
D) 测试失败时仍然合并，因为模型已经尝试修复

> **答案：A**  
> **解析：** 生产级 CI 集成需要非交互、结构化、可验证、可失败，而不只是生成建议文本。